In [ ]:
from lib import *
from lib_models import *
import numpy as np
import pandas as pd

In [ ]:
import torch
from torch.utils.data import DataLoader

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

In [ ]:
import pytorch_lightning as L 
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_lightning.loggers import CSVLogger

# load features from memmap & verify

In [ ]:
mm_eeg = load_eeg_memmap()

In [ ]:
g = plot_epoch_eeg(mm_eeg[102], start_epoch = 118, end_epoch = 123, n_epochs_to_plot = 3)

In [ ]:
mm_conn, S00_exams_indices, N_S00_exams  = load_s00_conn_memmap()

In [ ]:
mm_conn.shape

There are 1697 recordings from S00 (in the processed database). Each has 5 band-wise adjacency matrices of ffDTF metric.

In [ ]:
heatmap_bands(mm_conn[102][120])

# data utilities

## dataset

Let's treat the bands dimension as colors in RGB

Crate data splits with sklearn's [StratifiedGroupKFold](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedGroupKFold.html)

In [ ]:
meta = read_metadata()
labels = np.asarray(meta['pathology']).astype(int)[:N_S00_exams]
patients = np.asarray(meta['patient_id'])[:N_S00_exams]
sex = np.asarray(meta['female']).astype(int)[:N_S00_exams]
age = np.asarray(meta['age_group'])[:N_S00_exams]

In [ ]:
# stratify on label (+ optionally sex/age), group by patient so none is in both splits
#strata = labels.astype(str)                        # or: np.char.add(labels.astype(str), sex.astype(str))

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0)
folds_train_indieces = []
folds_val_indieces = []
for i, (train_index, test_index) in enumerate(sgkf.split(X = S00_exams_indices, 
                                             y = labels, 
                                             groups=patients)):
    print(f"Fold {i}:")
    print(f"  Train: index={train_index[:10]}")
    print(f"         group={patients[train_index][:10]}")
    print(f"  Test:  index={test_index[:10]}")
    print(f"         group={patients[test_index][:10]}")
    folds_train_indieces.append(train_index)
    folds_val_indieces.append(test_index)

In [ ]:
train_dataset = S00MILDataset(folds_train_indieces[0], max_instances=150,train=True)
val_dataset = S00MILDataset(folds_val_indieces[0], max_instances=150,train=False)

In [ ]:
train_loader = DataLoader(train_dataset, 
                          batch_size=10, 
                          shuffle=True,  
                          num_workers=10, 
                          pin_memory=True)

val_loader = DataLoader(val_dataset,  
                        batch_size=10, 
                        shuffle=False, 
                        num_workers=10, 
                        pin_memory=True)

## let's see if the model works

Quick sanity check before real training

In [ ]:
x, y = next(iter(train_loader))
print(x.shape, x.dtype)                      # expect (N, W, 5, 17, 17), float32
assert x.shape[-2:] == (17, 17)

model = BagEncoder(ConvAE2(latent=16))
model.eval()
with torch.no_grad():
    recon, z = model(x)
assert recon.shape == x.shape and z.shape == (x.shape[0], x.shape[1], 16)

Smoke test that runs one train & one val epoch (bugs discovery)

In [ ]:
trainer = L.Trainer(fast_dev_run=True, accelerator="auto")
trainer.fit(model, train_loader, val_loader)

Overfit a tiny subset

In [ ]:
trainer = L.Trainer(overfit_batches=2, 
                    max_epochs=100, 
                    accelerator="auto",
                    logger=CSVLogger("logs", name="conv_ae2"))
model.train()
trainer.fit(model, train_loader, val_loader)

## real training 

In [ ]:
ckpt = ModelCheckpoint(monitor="val_loss", 
                       mode="min", 
                       filename="ae2_fold0")
trainer = L.Trainer(
    max_epochs=150,
    accelerator="auto",
    logger=CSVLogger("logs", name="ae2_fold0"),
    callbacks=[EarlyStopping(monitor="val_loss", patience=10), ckpt],
)

model = BagEncoder(ConvAE2(latent=16))
trainer.fit(model, train_loader, val_loader)

In [ ]:
best = BagEncoder.load_from_checkpoint(ckpt.best_model_path, ae=ConvAE2(latent=16))
pred_trainer = L.Trainer(accelerator="auto", logger=False, enable_checkpointing=False)
out = pred_trainer.predict(best, val_loader) 

In [ ]:
recons  = torch.cat([r for r, _ in out]) #type:ignore       
latents = torch.cat([z for _, z in out]) #type:ignore  

In [ ]:
compare_heatmaps(val_ds = val_dataset, 
                 connectivity_mm = mm_conn, 
                 reconstructed = recons, 
                 recon_i = 5, 
                 window_i = 45)

In [ ]:
compare_heatmaps(val_ds = val_dataset, 
                 connectivity_mm = mm_conn, 
                 reconstructed = recons, 
                 recon_i = 50, 
                 window_i = 120)

In [ ]:
compare_heatmaps(val_ds = val_dataset, 
                 connectivity_mm = mm_conn, 
                 reconstructed = recons, 
                 recon_i = 333, 
                 window_i = 20)

In [ ]:
compare_heatmaps(val_ds = val_dataset, 
                 connectivity_mm = mm_conn, 
                 reconstructed = recons, 
                 recon_i = 13, 
                 window_i = 87)

It seems that the most basic convolution autoencoder is good for identifying source vertical lines. It shows the tendency to overpower pixels that originally have lower values.